# Fishbone - verification

Use the **Python (FAITH labelmaker)** kernel. Set `shot` and `window`, drag a
time range on any panel, then press *Mark present* / *Mark absent*, *Verify*
and *Save*.

The magnetic spectrogram. What you are looking for is a burst that chirps
**downward** through the 2-30 kHz band - the shaded region - repeating on the
beam-heated part of the discharge. A synthetic fishbone in this repository's
test fixtures sweeps 20 -> 12 kHz at -0.8 kHz/ms, which is the shape.

**The toroidal mode number is judged, not measured.** The corpus does not
record the MHR probes' toroidal angles, so the second panel shows the
cross-phase between a probe pair and leaves n = 1 to your eye. Nothing here
computes n.

Fishbones and sawtooth precursors are both the 1/1 kink and look alike on
magnetics. If a burst sits immediately before a crash in
`sawtooth_oscillation`, say so in the roster `notes`.

**No saved label grid exists for this category.** No detector has ever run
for fishbone, so `format/shots/` holds nothing and `review()` below warns
`no saved label grid` and leaves the label row out of the figure. That is
expected, not a bug: your corrections are the first labels this category
will have.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from scipy import signal as scipy_signal

from labeler.events.verify import Panel, corpus_signal, review

event = "fishbone"
shot = 192238  # replace with a shot from shots.csv
window = (1500.0, 3500.0)  # milliseconds
source = "format/shots"
probes = [0, 4]  # B1 and B5, the pair used for the cross-phase

mhr = corpus_signal(shot, "mhr", channels=probes, t_range=window)

# Take the rate from the SPAN, never from a median diff: xdata is float32 and
# its spacing quantises at t ~ 3 s.
rate = (mhr.x.shape[0] - 1) / ((mhr.x[-1] - mhr.x[0]) / 1000.0)
nperseg = 4096

freq, times, power = scipy_signal.spectrogram(
    mhr.y[0], fs=rate, nperseg=nperseg, noverlap=nperseg // 2
)
# Cross-phase needs each probe's own complex spectrum: phase(spec_a *
# conj(spec_b)) is the cross-spectrum's phase, which is the quantity that
# locks to a coherent n = 1 mode. The phase of a spectrogram taken of a
# complex signal built from the two real probes (a + i*b) is a different,
# meaningless quantity, so each probe is transformed on its own here.
_, _, spec_a = scipy_signal.spectrogram(
    mhr.y[0],
    fs=rate,
    nperseg=nperseg,
    noverlap=nperseg // 2,
    mode="complex",
)
_, _, spec_b = scipy_signal.spectrogram(
    mhr.y[1],
    fs=rate,
    nperseg=nperseg,
    noverlap=nperseg // 2,
    mode="complex",
)
cross_phase = np.angle(spec_a * np.conj(spec_b))
keep = freq <= 40000.0

panels = [
    Panel(
        title=f"mhr B{probes[0] + 1} spectrogram",
        kind="heatmap",
        x=times * 1000.0 + mhr.x[0],
        y=freq[keep] / 1000.0,
        z=np.log10(power[keep] + 1e-30),
        ylabel="kHz",
        bands=[(2.0, 30.0)],
    ),
    Panel(
        title=f"cross-phase B{probes[0] + 1} x B{probes[1] + 1}",
        kind="heatmap",
        x=times * 1000.0 + mhr.x[0],
        y=freq[keep] / 1000.0,
        z=cross_phase[keep],
        ylabel="kHz",
        bands=[(2.0, 30.0)],
    ),
]

In [ ]:
session = review(event, shot, panels, source=source)
session

This category has no formatted labels yet, so the label row will be
missing and `review()` simply leaves it out. Your corrections are the first
annotation it has.

```python
from labeler.events.verify import read_corrections, review_path
read_corrections(review_path(event, shot))
```
